# Catalyst on TPU: smoke test

1. **Runtime → Change runtime type → a TPU** (e.g. v5e-1). A single-device TPU still runs the multi-process tests, with one process.
2. **Runtime → Run all.** Every test runs in its own subprocess, because only one process can hold the TPU.
3. **Copy the output of the last cell back to Claude.**

| Test | What it runs | Expected |
|---|---|---|
| `single` | `dl.Engine()` on one XLA device: loss, accuracy, `CheckpointCallback` (`xm.save`), `load_best_on_end` | PASS |
| `ddp` | `dl.DistributedXLAEngine()`: `xmp.spawn(nprocs=None)`, metrics averaged across processes (`xm.mesh_reduce`), collective `xm.save` | PASS |
| `auc` | `single` plus `AUCCallback` | FAIL on torch_xla ≥ 2.1 (`TORCH_DIST_ROOT`, the break that is only commented in `utils/distributed.py`) |
| `cifar`, `cifar_ddp` | ResNet9 from catalyst's XLA notebooks, CIFAR10 subset, 1 epoch | PASS |

In [ ]:
# catalyst to test; switch to neuroneural/catalyst master once the branch is merged
REPO, BRANCH = "paavalipopov/catalyst", "accelerate-xla"
CATALYST = f"git+https://github.com/{REPO}@{BRANCH}"
# accelerate versions to test; add "0.27.2" for the pre-0.28 `TPU` name
# (it may fail inside accelerate itself on new torch_xla)
ACCELERATE_VERSIONS = ["latest", "0.34.2"]
TESTS = ["single", "ddp", "auc", "cifar", "cifar_ddp"]
TIMEOUT = 1800  # seconds per test

In [ ]:
# environment, probed in a subprocess so this kernel never grabs the TPU
import json, os, subprocess, sys

PROBE = r"""
import json, os, sys, torch
info = {"python": sys.version.split()[0], "torch": torch.__version__}
try:
    import torch_xla, torch_xla.runtime as xr
    info["torch_xla"] = torch_xla.__version__
    info["device_type"] = xr.device_type()
    info["devices"] = xr.global_runtime_device_count()
except Exception as ex:
    info["torch_xla_error"] = f"{type(ex).__name__}: {ex}"
print(json.dumps(info))
"""
probe = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
try:
    env_info = json.loads(probe.stdout.strip().splitlines()[-1])
except (IndexError, json.JSONDecodeError):
    env_info = {"probe_error": probe.stderr.strip().splitlines()[-3:]}
print(json.dumps(env_info, indent=1))
if "torch_xla" not in env_info and os.environ.get("SMOKE_DEVICE") != "cpu":
    raise RuntimeError(
        "torch_xla is not usable here. Check that the runtime type is a TPU; if it is, install a torch_xla "
        "matching torch: pip install 'torch_xla[tpu]==<torch version>' "
        "-f https://storage.googleapis.com/libtpu-releases/index.html"
    )

In [ ]:
# catalyst from the branch, without letting pip touch torch / torch_xla
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hydra-slayer", "tqdm", "tensorboardX"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", CATALYST], check=True)
print(subprocess.run([sys.executable, "-c", "import catalyst; print('catalyst', catalyst.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
%%writefile catalyst_tpu_smoke.py
"""Catalyst TPU smoke tests. One test per process: python catalyst_tpu_smoke.py <test>.

Tests: single (dl.Engine on one XLA device), ddp (dl.DistributedXLAEngine over all devices),
auc (single + AUCCallback), cifar / cifar_ddp (ResNet9 from the catalyst XLA notebooks, CIFAR10 subset).
Prints one line `RESULT {json}`. SMOKE_DEVICE=cpu swaps in CPU engines for a local dry run.
"""
import json
import os
import sys
import tempfile
import traceback

TEST = sys.argv[1]
DEVICE = os.environ.get("SMOKE_DEVICE", "tpu")
if DEVICE == "tpu":
    os.environ.setdefault("PJRT_DEVICE", "TPU")

import torch  # noqa: E402
from torch import nn  # noqa: E402
from torch.utils.data import DataLoader, Subset, TensorDataset  # noqa: E402

from catalyst import dl  # noqa: E402

DISTRIBUTED = TEST.endswith("ddp")


def report(status, **info):
    print("RESULT " + json.dumps({"test": TEST, "status": status, **info}), flush=True)


def make_engine():
    if DEVICE == "cpu":
        if DISTRIBUTED:
            return dl.DistributedDataParallelEngine(
                port=29561, num_node_workers=2, process_group_kwargs={"backend": "gloo"}, cpu=True
            )
        return dl.CPUEngine()
    return dl.DistributedXLAEngine() if DISTRIBUTED else dl.Engine()


def resnet9(in_channels=3, num_classes=10, size=16):
    # same model as catalyst's XLA notebooks
    from catalyst.contrib.layers import ResidualBlock

    def block(cin, cout, pool=False):
        layers = [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        return nn.Sequential(*layers, *([nn.MaxPool2d(2)] if pool else []))

    s1, s2, s4, s8 = size, size * 2, size * 4, size * 8
    return nn.Sequential(
        block(in_channels, s1),
        block(s1, s2, pool=True),
        ResidualBlock(nn.Sequential(block(s2, s2), block(s2, s2))),
        block(s2, s4, pool=True),
        block(s4, s8, pool=True),
        ResidualBlock(nn.Sequential(block(s8, s8), block(s8, s8))),
        nn.Sequential(nn.MaxPool2d(4), nn.Flatten(), nn.Dropout(0.2), nn.Linear(s8, num_classes)),
    )


def cifar_loaders():
    if DEVICE == "cpu":  # dry run: CIFAR-shaped random data, no download
        x, y = torch.randn(256, 3, 32, 32), torch.randint(0, 10, (256,))
        return DataLoader(TensorDataset(x, y), batch_size=64), DataLoader(TensorDataset(x, y), batch_size=64)
    from catalyst.contrib.data import Compose, ImageToTensor, NormalizeImage
    from catalyst.contrib.datasets import CIFAR10

    tf = Compose([ImageToTensor(), NormalizeImage((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    root = os.path.expanduser("~/cifar")
    train = CIFAR10(root, train=True, download=True, transform=tf)
    valid = CIFAR10(root, train=False, download=True, transform=tf)
    # subsets keep the run to a few minutes
    train, valid = Subset(train, range(4096)), Subset(valid, range(1024))
    return DataLoader(train, batch_size=64, shuffle=True), DataLoader(valid, batch_size=64)


class SmokeRunner(dl.IRunner):
    def __init__(self, logdir):
        super().__init__()
        self._logdir = logdir

    @property
    def num_epochs(self):
        return 1 if TEST.startswith("cifar") else 2

    def get_engine(self):
        return make_engine()

    def get_loggers(self):
        return {"csv": dl.CSVLogger(logdir=self._logdir)}

    def get_loaders(self):
        if TEST.startswith("cifar"):
            train, valid = cifar_loaders()
            return {"train": train, "valid": valid}
        g = torch.Generator().manual_seed(0)
        x = torch.randn(512, 16, generator=g)
        y = (x[:, 0] > 0).long() + 2 * (x[:, 1] > 0).long()
        data = TensorDataset(x, y)
        return {
            "train": DataLoader(data, batch_size=32, shuffle=True),
            "valid": DataLoader(data, batch_size=64),
        }

    def get_model(self):
        torch.manual_seed(0)
        if TEST.startswith("cifar"):
            return resnet9()
        return nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 4))

    def get_criterion(self):
        return nn.CrossEntropyLoss()

    def get_optimizer(self, model):
        return torch.optim.Adam(model.parameters(), lr=1e-2)

    def get_scheduler(self, optimizer):
        return None

    def get_callbacks(self):
        callbacks = {
            "criterion": dl.CriterionCallback(
                input_key="logits", target_key="targets", metric_key="loss"
            ),
            "backward": dl.BackwardCallback(metric_key="loss"),
            "optimizer": dl.OptimizerCallback(metric_key="loss"),
            "accuracy": dl.AccuracyCallback(input_key="logits", target_key="targets", topk=(1,)),
            "checkpoint": dl.CheckpointCallback(
                self._logdir,
                loader_key="valid",
                metric_key="loss",
                minimize=True,
                topk=1,
                load_best_on_end=True,
            ),
        }
        if TEST == "auc":
            callbacks["auc"] = dl.AUCCallback(input_key="logits", target_key="targets")
        return callbacks

    def handle_batch(self, batch):
        x, y = batch
        self.batch = {"features": x, "targets": y, "logits": self.model(x)}

    def on_experiment_end(self, runner):
        dtype = self.engine.distributed_type
        info = {
            "rank": int(self.engine.process_index),
            "world_size": int(self.engine.num_processes),
            "distributed_type": getattr(dtype, "value", str(dtype)),
            "device": str(self.engine.device),
            "valid": {k: float(v) for k, v in self.epoch_metrics["valid"].items()},
        }
        with open(os.path.join(self._logdir, f"rank{info['rank']}.json"), "w") as fout:
            json.dump(info, fout)
        super().on_experiment_end(runner)


def main():
    logdir = tempfile.mkdtemp(prefix=f"smoke_{TEST}_")
    try:
        SmokeRunner(logdir).run()

        # every rank reports; DDP-averaged metrics must agree across ranks
        files = sorted(f for f in os.listdir(logdir) if f.startswith("rank"))
        infos = [json.load(open(os.path.join(logdir, f))) for f in files]
        assert infos, "no rank wrote a report"
        world_size = infos[0]["world_size"]
        assert len(infos) == world_size, f"{len(infos)} rank reports for world_size {world_size}"
        losses = {round(i["valid"]["loss"], 5) for i in infos}
        assert len(losses) == 1, f"ranks disagree on the valid loss: {sorted(losses)}"
        assert os.path.isfile(os.path.join(logdir, "model.best.pth")), "no model.best.pth"

        report(
            "PASS",
            world_size=world_size,
            distributed_type=infos[0]["distributed_type"],
            device=infos[0]["device"],
            valid={k: round(v, 4) for k, v in infos[0]["valid"].items()},
        )
    except BaseException as ex:  # noqa: B902 - report every failure, including SystemExit from workers
        lines = traceback.format_exc().strip().splitlines()
        report("FAIL", error=f"{type(ex).__name__}: {ex}"[:600], traceback_tail=lines[-6:])
        sys.exit(1)


if __name__ == "__main__":
    main()


In [ ]:
# run every test under every accelerate version
results = []
for acc in ACCELERATE_VERSIONS:
    spec = "accelerate" if acc == "latest" else f"accelerate=={acc}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", spec], check=True)
    version = subprocess.run([sys.executable, "-c", "import accelerate; print(accelerate.__version__)"],
                             capture_output=True, text=True).stdout.strip()

    for test in TESTS:
        print(f"accelerate {version} | {test} ...", flush=True)
        try:
            proc = subprocess.run([sys.executable, "catalyst_tpu_smoke.py", test],
                                  capture_output=True, text=True, timeout=TIMEOUT)
            lines = [l for l in proc.stdout.splitlines() if l.startswith("RESULT ")]
            if lines:
                res = json.loads(lines[-1][len("RESULT "):])
            else:
                res = {"test": test, "status": "FAIL", "error": "no RESULT line",
                       "stderr_tail": proc.stderr.strip().splitlines()[-8:]}
        except subprocess.TimeoutExpired:
            res = {"test": test, "status": "FAIL", "error": f"timeout after {TIMEOUT}s"}

        res["accelerate"] = version
        if res["status"] == "FAIL" and "TORCH_DIST_ROOT" in json.dumps(res):
            res["note"] = "known: utils/distributed.py needs torch_xla.runtime (commented, not fixed)"
        results.append(res)
        print("   ->", res["status"], "" if res["status"] == "PASS" else res.get("error", "")[:300])

In [ ]:
# summary: copy everything below back to Claude
print(f"{'accelerate':<12} {'test':<10} {'status':<6} world  info")
for r in results:
    if r["status"] == "FAIL":
        info = r.get("note") or r.get("error", "")
    else:
        info = r.get("distributed_type", "")
    print(f"{r['accelerate']:<12} {r['test']:<10} {r['status']:<6} {str(r.get('world_size', '-')):<6} {info[:120]}")
print()
print(json.dumps({"env": env_info, "results": results}))